In [1]:
!pip install pyvi pycocoevalcap pandas

In [2]:
import json
import os
import pandas as pd
from pyvi import ViTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.rouge.rouge import Rouge
from collections import defaultdict

# --- 1. CẤU HÌNH ĐƯỜNG DẪN FILE ---
GT_FILE = "test_groundtruth.jsonl"
PRED_FILES = {
    "Qwen2-VL-Base": "results_qwen2vl_test.jsonl",
    "Qwen2-VL-FT-V1 (step 4000)": "results_qwen2vl_test_finetuned_v1.jsonl",
    "Qwen2-VL-FT-V2 (step 6000)": "results_qwen2vl_test_finetuned_v2.jsonl"
}

d:\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [3]:
# --- 2. HÀM TIỀN XỬ LÝ VÀ LOAD DATA ---
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def preprocess_vn(text):
    if not text: return ""
    return ViTokenizer.tokenize(text.lower())

# A. Load Groundtruth và gom nhóm (vì GT lộn xộn và 1 ảnh có 5 câu)
print("📂 Đang nạp Groundtruth...")
gt_raw = load_jsonl(GT_FILE)
references = defaultdict(list)
for item in gt_raw:
    img_id = item['file_name']
    references[img_id].append(preprocess_vn(item['caption']))

# B. Hàm tính toán Metrics cho một file dự đoán
def eval_model(pred_path):
    preds_raw = load_jsonl(pred_path)
    
    gts = {}
    res = {}
    
    for item in preds_raw:
        img_id = item['file_name']
        if img_id in references:
            # Tokenize dự đoán
            res[img_id] = [preprocess_vn(item['prediction'])]
            # Lấy nhãn tương ứng đã tokenize sẵn
            gts[img_id] = references[img_id]
            
    # Khởi tạo bộ chấm điểm
    scorers = [
        (Bleu(4), ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4"]),
        (Cider(), "CIDEr"),
        (Rouge(), "ROUGE-L")
    ]
    
    results = {}
    for scorer, method in scorers:
        score, _ = scorer.compute_score(gts, res)
        if isinstance(method, list):
            for m, s in zip(method, score):
                results[m] = round(s * 100, 2)
        else:
            results[method] = round(score * 100, 2)
    return results

📂 Đang nạp Groundtruth...


In [5]:
# --- 3. THỰC THI ĐÁNH GIÁ TẤT CẢ MODEL ---
all_metrics = {}

for model_name, path in PRED_FILES.items():
    print(f"📊 Đang chấm điểm cho: {model_name}...")
    all_metrics[model_name] = eval_model(path)

# --- 4. BỔ SUNG DỮ LIỆU HIỆU SUẤT (Dựa trên hình ảnh bạn cung cấp) ---
efficiency_data = {
    "Qwen2-VL-Base": {
        "Time/Img (s)": 2.8127,
        "VRAM (GB)": 4.23,
        "Params (B)": 2.21,
        "Disk (GB)": 4.13
    },
    "Qwen2-VL-FT-V1 (step 4000)": {
        "Time/Img (s)": 2.4891,
        "VRAM (GB)": 4.23,
        "Params (B)": 2.21,
        "Disk (GB)": 4.13
    },
    "Qwen2-VL-FT-V2 (step 6000)": {
        "Time/Img (s)": 2.5667,
        "VRAM (GB)": 4.23,
        "Params (B)": 2.21,
        "Disk (GB)": 4.13
    }
}


# Kết hợp Metrics và Efficiency
final_comparison = {}
for name in PRED_FILES.keys():
    final_comparison[name] = {**all_metrics[name], **efficiency_data[name]}

# --- 5. XUẤT BẢNG TỔNG HỢP ---
df = pd.DataFrame(final_comparison).T

# Sắp xếp cột cho đúng thứ tự mong muốn
cols = ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4", "CIDEr", "ROUGE-L", "Time/Img (s)", "VRAM (GB)", "Params (B)"]
df = df[cols]

print("\n" + "="*100)
print("🏆 BẢNG SO SÁNH HIỆU NĂNG TRƯỚC VÀ SAU FINE-TUNING (WORD-LEVEL)")
print("="*100)
display(df)
print("="*100)

📊 Đang chấm điểm cho: Qwen2-VL-Base...
{'testlen': 31694, 'reflen': 26892, 'guess': [31694, 30894, 30094, 29294], 'correct': [12026, 2210, 425, 73]}
ratio: 1.1785661163170764
📊 Đang chấm điểm cho: Qwen2-VL-FT-V1 (step 4000)...
{'testlen': 24969, 'reflen': 25267, 'guess': [24969, 24169, 23369, 22569], 'correct': [17034, 7813, 4012, 2324]}
ratio: 0.988205960343492
📊 Đang chấm điểm cho: Qwen2-VL-FT-V2 (step 6000)...
{'testlen': 25206, 'reflen': 25283, 'guess': [25206, 24406, 23606, 22806], 'correct': [16879, 7656, 3960, 2268]}
ratio: 0.9969544753391213

🏆 BẢNG SO SÁNH HIỆU NĂNG TRƯỚC VÀ SAU FINE-TUNING (WORD-LEVEL)


,BLEU-1,BLEU-2,BLEU-3,BLEU-4,CIDEr,ROUGE-L,Time/Img (s),VRAM (GB),Params (B)
Qwen2-VL-Base,37.94,16.48,7.26,3.13,4.52,17.66,2.8127,4.23,2.21
Qwen2-VL-FT-V1 (step 4000),67.41,46.40,33.18,24.69,47.52,39.56,2.4891,4.23,2.21
Qwen2-VL-FT-V2 (step 6000),66.76,45.69,32.68,24.26,46.22,38.99,2.5667,4.23,2.21
